# 백엔드 개발자 취업 준비 로드맵

> 목표: 현재 게시판 프로젝트를 확장하면서 DB·API·테스트·배포 역량을 단계적으로 갖춥니다.

## 학습 방법
1. 아래 순서대로 한 주제씩 공부합니다.
2. 개념을 읽은 뒤, 현재 프로젝트에 작은 기능으로 직접 적용합니다.
3. 적용 결과와 배운 점을 README에 기록합니다.

## 학습 순서
1. 데이터베이스 심화
2. API 설계
3. 테스트 코드
4. CI/CD
5. AI 모델 연동

---

## 1. 데이터베이스 심화 학습

### 1-1. 학습 순서 (기본 구조만 아는 상태 → 심화)

1. **SQL 기본기 재점검**: SELECT/JOIN/GROUP BY를 손으로 짤 수 있는지 확인
2. **정규화 (1NF~3NF)**: 왜 테이블을 나누는지, 언제 일부러 비정규화하는지
3. **인덱스**: B-Tree 구조 이해, 어떤 컬럼에 인덱스를 걸어야 하는지, `EXPLAIN`으로 쿼리 분석
4. **트랜잭션**: ACID, 격리 수준(Isolation Level), 락(Lock)
5. **N+1 문제**: ORM 쓸 때 가장 흔한 성능 문제
6. **커넥션 풀**: 동시 요청 많을 때 DB 연결을 어떻게 재사용하는지

### 1-2. 실습 예제 — N+1 문제와 해결 (SQLAlchemy 기준)

```python
# 나쁜 예 (N+1 발생) - 유저 100명 조회하면 쿼리 101번 실행됨
users = session.query(User).all()
for user in users:
    print(user.posts)  # 매번 별도 쿼리 발생!

# 좋은 예 - JOIN으로 한 번에 가져오기
from sqlalchemy.orm import joinedload

users = session.query(User).options(joinedload(User.posts)).all()
for user in users:
    print(user.posts)  # 추가 쿼리 없음
```

### 1-3. 회원가입/인증 시스템에 바로 적용할 DB 설계 팁

인증번호 테이블을 별도로 분리하는 걸 추천합니다:

```sql
CREATE TABLE verification_codes (
    id SERIAL PRIMARY KEY,
    email VARCHAR(255) NOT NULL,
    code_hash VARCHAR(255) NOT NULL,  -- 평문 저장 금지, 해싱
    expires_at TIMESTAMP NOT NULL,
    attempt_count INT DEFAULT 0,
    created_at TIMESTAMP DEFAULT NOW()
);
CREATE INDEX idx_verification_email ON verification_codes(email);
```

이유: users 테이블에 인증번호를 직접 넣으면 만료된 데이터가 계속 쌓이고, 조회 성능도 떨어집니다. 별도 테이블 + 주기적 삭제(cron)가 실무 패턴입니다.

### 1-4. 추천 학습 자료
- "SQL 첫걸음" (기초) → "Real MySQL 8.0" (심화, 실무 필독서)
- 직접 EXPLAIN ANALYZE 찍어보면서 느린 쿼리 찾는 연습

---

## 2. API 설계 심화

### 2-1. REST 설계 원칙 체크리스트

| 항목 | 나쁜 예 | 좋은 예 |
|---|---|---|
| URL 명명 | `/getUser`, `/user_list` | `/users`, `/users/{id}` |
| HTTP 메서드 | 모든 요청을 POST로 | GET(조회)/POST(생성)/PUT,PATCH(수정)/DELETE(삭제) |
| 상태 코드 | 항상 200만 반환 | 201(생성), 400(잘못된 요청), 401(인증 필요), 404(없음), 500(서버 에러) |
| 응답 형식 | 매번 다른 구조 | 일관된 envelope 구조 |

### 2-2. 표준 응답 포맷 예시

```python
# 성공 응답
{
    "success": True,
    "data": {"id": 1, "email": "user@example.com"},
    "message": "회원가입이 완료되었습니다."
}

# 실패 응답 (항상 같은 구조 유지)
{
    "success": False,
    "error": {
        "code": "INVALID_VERIFICATION_CODE",
        "message": "인증번호가 일치하지 않습니다."
    }
}
```

이렇게 일관된 구조를 쓰면 프론트엔드(또는 API를 쓰는 누구든)가 매번 다르게 파싱할 필요가 없어져서 실무에서 매우 중요하게 봅니다.

### 2-3. API 버저닝

```
/api/v1/users
/api/v2/users
```
서비스가 커지면 기존 클라이언트를 깨뜨리지 않고 API를 바꿔야 할 때가 옵니다. 처음부터 `/v1/`을 붙여두는 습관을 들이세요.

### 2-4. OpenAPI(Swagger) 문서화

FastAPI를 쓰신다면 자동으로 문서가 생성됩니다:
```python
from fastapi import FastAPI

app = FastAPI(title="My Backend API", version="1.0.0")

@app.post("/api/v1/auth/verify", summary="이메일 인증번호 확인")
async def verify_code(email: str, code: str):
    """
    - **email**: 인증할 이메일 주소
    - **code**: 사용자가 입력한 6자리 인증번호
    """
    ...
```
`/docs` 경로로 가면 자동 문서 페이지가 뜹니다. 면접에서 "API 문서화 어떻게 하셨어요?"라는 질문에 바로 보여줄 수 있는 포인트예요.

---

## 3. CI/CD 파이프라인

### 3-1. 개념 정리

- **CI (Continuous Integration)**: 코드를 푸시할 때마다 자동으로 테스트/빌드를 돌려서 문제를 빨리 발견
- **CD (Continuous Deployment/Delivery)**: 테스트를 통과하면 자동으로 서버에 배포

지금은 Docker로 수동 배포하고 계신 것 같은데, 다음 단계는 "push하면 자동으로 테스트→빌드→배포"까지 이어지는 흐름입니다.

### 3-2. GitHub Actions 예시 (Python + Docker 기준)

`.github/workflows/deploy.yml`:

```yaml
name: CI/CD Pipeline

on:
  push:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Python 설치
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - name: 의존성 설치
        run: pip install -r requirements.txt
      - name: 테스트 실행
        run: pytest --cov=app tests/

  build-and-deploy:
    needs: test  # 테스트 통과해야만 배포 진행
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Docker 이미지 빌드
        run: docker build -t my-backend:${{ github.sha }} .
      - name: 서버에 배포
        run: |
          echo "여기에 서버 접속 후 새 이미지로 교체하는 스크립트"
```

핵심 포인트: **테스트가 실패하면 배포 자체가 진행되지 않도록** `needs: test`로 묶는 것. 이게 파이프라인의 핵심 가치입니다.

### 3-3. 학습 순서
1. GitHub Actions로 "push하면 pytest 자동 실행"부터 연습
2. 성공하면 "테스트 통과 시 Docker 이미지 빌드"까지 추가
3. 마지막에 실제 서버(EC2 등)에 SSH로 접속해서 배포까지 자동화

---

## 4. 테스트 코드 (pytest 기준, 실무 패턴 위주로 다수 제공)

### 4-1. 기본 단위 테스트 — 회원가입 로직

```python
# app/services/auth.py
def hash_password(password: str) -> str:
    import bcrypt
    return bcrypt.hashpw(password.encode(), bcrypt.gensalt()).decode()

def verify_password(password: str, hashed: str) -> bool:
    import bcrypt
    return bcrypt.checkpw(password.encode(), hashed.encode())
```

```python
# tests/test_auth.py
from app.services.auth import hash_password, verify_password

def test_hash_password_creates_different_hash_each_time():
    """같은 비밀번호도 매번 다른 해시가 나와야 함 (salt 때문)"""
    hash1 = hash_password("mypassword123")
    hash2 = hash_password("mypassword123")
    assert hash1 != hash2

def test_verify_password_with_correct_password():
    hashed = hash_password("mypassword123")
    assert verify_password("mypassword123", hashed) is True

def test_verify_password_with_wrong_password():
    hashed = hash_password("mypassword123")
    assert verify_password("wrongpassword", hashed) is False
```

### 4-2. 인증번호 만료 로직 테스트 (시간 관련 — 실무에서 자주 나옴)

```python
# app/services/verification.py
from datetime import datetime, timedelta

def is_code_expired(created_at: datetime, ttl_minutes: int = 5) -> bool:
    return datetime.now() > created_at + timedelta(minutes=ttl_minutes)
```

```python
# tests/test_verification.py
from datetime import datetime, timedelta
from app.services.verification import is_code_expired
from freezegun import freeze_time  # pip install freezegun

def test_code_not_expired_within_ttl():
    created = datetime.now()
    assert is_code_expired(created, ttl_minutes=5) is False

def test_code_expired_after_ttl():
    created = datetime.now() - timedelta(minutes=10)
    assert is_code_expired(created, ttl_minutes=5) is True

@freeze_time("2026-01-01 12:00:00")
def test_code_expiry_with_frozen_time():
    """시간을 고정시켜서 정확히 경계값 테스트"""
    created = datetime.now()
    with freeze_time("2026-01-01 12:04:59"):
        assert is_code_expired(created, ttl_minutes=5) is False
    with freeze_time("2026-01-01 12:05:01"):
        assert is_code_expired(created, ttl_minutes=5) is True
```

### 4-3. Mocking — 이메일 발송처럼 "외부 의존성"이 있는 경우

실제로 매 테스트마다 이메일을 보내면 안 되니까 **mock으로 대체**합니다.

```python
# app/services/email.py
def send_verification_email(email: str, code: str):
    # 실제로는 SMTP 서버 호출
    ...

# app/services/signup.py
from app.services.email import send_verification_email

def signup_user(email: str, password: str, db_session):
    code = generate_code()
    save_verification_code(db_session, email, code)
    send_verification_email(email, code)  # 외부 의존성
    return {"success": True}
```

```python
# tests/test_signup.py
from unittest.mock import patch
from app.services.signup import signup_user

@patch("app.services.signup.send_verification_email")
def test_signup_sends_verification_email(mock_send_email, db_session):
    signup_user("test@example.com", "password123", db_session)

    # 이메일 발송 함수가 정확히 호출됐는지만 확인 (실제 이메일 안 보냄)
    mock_send_email.assert_called_once()
    call_args = mock_send_email.call_args[0]
    assert call_args[0] == "test@example.com"

@patch("app.services.signup.send_verification_email")
def test_signup_creates_verification_code_in_db(mock_send_email, db_session):
    signup_user("test@example.com", "password123", db_session)

    saved_code = db_session.query(VerificationCode).filter_by(
        email="test@example.com"
    ).first()
    assert saved_code is not None
```

### 4-4. Fixture — 반복되는 준비 코드 재사용

```python
# tests/conftest.py
import pytest
from app.database import get_test_db_session

@pytest.fixture
def db_session():
    """각 테스트마다 깨끗한 DB 세션 제공, 끝나면 롤백"""
    session = get_test_db_session()
    yield session
    session.rollback()
    session.close()

@pytest.fixture
def test_user(db_session):
    """테스트용 유저를 미리 만들어두는 fixture"""
    user = User(email="fixture@example.com", password_hash="dummy")
    db_session.add(user)
    db_session.commit()
    return user
```

```python
# tests/test_login.py
def test_login_with_existing_user(db_session, test_user):
    # test_user fixture 덕분에 유저 생성 코드를 매번 안 써도 됨
    result = login(db_session, "fixture@example.com", "password123")
    assert result is not None
```

### 4-5. API 엔드포인트 통합 테스트 (FastAPI 기준)

```python
# tests/test_api_signup.py
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

def test_signup_endpoint_success():
    response = client.post("/api/v1/auth/signup", json={
        "email": "newuser@example.com",
        "password": "SecurePass123!"
    })
    assert response.status_code == 201
    assert response.json()["success"] is True

def test_signup_endpoint_with_duplicate_email(test_user):
    response = client.post("/api/v1/auth/signup", json={
        "email": "fixture@example.com",  # 이미 존재하는 이메일
        "password": "SecurePass123!"
    })
    assert response.status_code == 400
    assert response.json()["error"]["code"] == "EMAIL_ALREADY_EXISTS"

def test_verify_code_with_wrong_code():
    response = client.post("/api/v1/auth/verify", json={
        "email": "test@example.com",
        "code": "000000"
    })
    assert response.status_code == 400
```

### 4-6. 커버리지 확인
```bash
pip install pytest-cov
pytest --cov=app --cov-report=html tests/
```
`htmlcov/index.html`을 열면 코드에서 테스트가 안 된 부분을 시각적으로 볼 수 있습니다. 실무에서는 보통 70~80% 커버리지를 최소선으로 잡는 경우가 많습니다.

---

## 5. AI 모델 연동 심화 분석

이미 PyTorch 모델을 다운로드해서 연동하신 상태니, 여기서는 **더 깊게 이해하면 좋은 부분**을 정리합니다.

### 5-1. 지금 구조의 잠재적 문제점 체크

| 항목 | 확인할 것 |
|---|---|
| 모델 로드 시점 | 요청마다 로드하면 매우 느림 → 서버 시작 시 **한 번만** 로드해야 함 |
| 동시 요청 처리 | 여러 요청이 동시에 오면 GPU 메모리 충돌 가능 → 큐(Queue) 또는 세마포어로 제어 |
| 메모리 관리 | 추론 후 `torch.cuda.empty_cache()` 필요한 경우도 있음 |

### 5-2. 모델을 서버 시작 시 한 번만 로드하는 패턴 (FastAPI)

```python
from fastapi import FastAPI
import torch

app = FastAPI()
model = None  # 전역 변수

@app.on_event("startup")
def load_model():
    global model
    model = torch.load("model.pt", map_location="cpu")
    model.eval()
    print("모델 로드 완료")

@app.post("/predict")
async def predict(input_data: dict):
    with torch.no_grad():  # 추론 시 gradient 계산 끄기 (속도/메모리 절약)
        output = model(preprocess(input_data))
    return {"result": postprocess(output)}
```

`with torch.no_grad()`를 빼먹으면 추론인데도 학습용 메모리를 잡아먹어서 느려집니다. 이거 한 줄이 실무에서 성능 차이를 크게 만듭니다.

### 5-3. 동시 요청이 몰릴 때 — 배치 처리 패턴

요청이 하나씩 올 때마다 모델을 돌리면 비효율적입니다. 짧은 시간 동안 요청을 모아서 한 번에 배치로 처리하는 게 실무 패턴입니다 (예: `TorchServe`, 또는 직접 큐 구현).

### 5-4. 배포 시 가벼운 대안 — ONNX 변환

PyTorch 자체는 무겁고 느립니다. 실서비스 배포 시에는 ONNX로 변환하는 걸 고려해볼 수 있어요:

```python
import torch

# PyTorch 모델을 ONNX로 변환
dummy_input = torch.randn(1, 3, 224, 224)  # 입력 형태에 맞게 조정
torch.onnx.export(model, dummy_input, "model.onnx")
```

이후 서버에서는 `onnxruntime`만 설치하면 되고, PyTorch 전체를 설치할 필요가 없어져서 이미지가 훨씬 가벼워집니다.

### 5-5. AI 모델 연동 테스트 코드

```python
# tests/test_model_inference.py
from unittest.mock import patch, MagicMock

@patch("app.main.model")
def test_predict_endpoint_returns_expected_shape(mock_model):
    mock_model.return_value = MagicMock()
    # 실제 무거운 모델 대신 가짜 모델로 API 로직만 테스트
    response = client.post("/predict", json={"input": [1, 2, 3]})
    assert response.status_code == 200
    assert "result" in response.json()
```

실제 모델을 테스트마다 로드하면 테스트가 너무 느려지므로, API 로직 테스트에서는 모델 자체를 mock 처리하는 게 일반적입니다.

---

## 6. 정리 — 다음 4주 학습 순서 제안

| 주차 | 목표 |
|---|---|
| 1주차 | DB 인덱스/트랜잭션/N+1 이해 + 지금 프로젝트 쿼리 최적화 |
| 2주차 | API 응답 포맷 표준화 + Swagger 문서화 적용 |
| 3주차 | 지금까지 만든 기능에 pytest로 테스트 코드 전부 작성 (커버리지 70%+ 목표) |
| 4주차 | GitHub Actions로 CI 파이프라인 구축 (테스트 자동 실행부터) |

지금 진행 중인 프로젝트 하나에 이 내용들을 하나씩 적용해가면서, 완성되면 그 자체가 훌륭한 포트폴리오이자 면접 답변 소스가 됩니다.